In [ ]:
from src.utils.config import load_config, resolve_path
from src.features.build_features import build_feature_table
cfg = load_config()
tx = pd.read_parquet(resolve_path(cfg.get('paths.interim_data')))
table = build_feature_table(tx, str(cfg.get('features.snapshot_date')),
    int(cfg.get('features.churn_window_days')),
    int(cfg.get('features.clv_window_days')))
table.describe().T

In [ ]:
# Leakage spot-check: features use pre-cutoff data only.
# (Pinned permanently by tests/test_features.py.)
from src.features.build_features import split_by_cutoff
history, future = split_by_cutoff(tx, str(cfg.get('features.snapshot_date')))
assert (history['invoice_date'] < future['invoice_date'].min()) or future.empty
print('cutoff windows disjoint: OK')

In [ ]:
# Feature-target correlation snapshot
corr = table.drop(columns=['customer_id', 'country']).corr(numeric_only=True)['churn'].sort_values()
corr.head(12), corr.tail(12)